# LOCO YOLO11 training, seed 0 (all folds)
Settings: Accelerator = GPU, Internet = On. Then **Save Version > Save & Run All (Commit)**; runs in the background. Download `results_seeds_0.zip` from the Output tab into `outputs/kaggle/`.

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.170"], check=True)
cands = [p for d in ("*", "*/*", "*/*/*") for p in glob.glob(f"/kaggle/input/{d}/manifest.csv")]
assert len(cands) == 1, f"expected one manifest.csv under /kaggle/input, found {cands}"
DS = str(Path(cands[0]).parent)
print("dataset:", DS, os.listdir(DS))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


In [ ]:
%%writefile /kaggle/working/export_yolo.py
"""Build a YOLO-format dataset for one LOCO fold (read-only on data/raw, manifest, splits).

  python scripts/export_yolo.py --fold C4 --images-root <dir holding data_C1/ ... data_C6/> --out <dir> [--link|--copy]

Layout: <out>/images/{train,val,cal,target}/<name>, <out>/labels/<role>/<stem>.txt, <out>/data.yaml, <out>/index.csv.
name = manifest path with '/' -> '__' (unique across folders). One class: polyp.
Boxes: gt_boxes ([x1,y1,x2,y2], x2/y2 exclusive) clipped to the image, written as normalized cx cy w h.
"""
import argparse, json, os, shutil, sys
from pathlib import Path
import pandas as pd

ROLES = ["train", "val", "cal", "target"]
ROOT = Path(__file__).resolve().parents[1]


def yolo_lines(boxes, w, h):
    lines, dropped = [], 0
    for x1, y1, x2, y2 in boxes:
        x1, x2 = max(0, min(w, x1)), max(0, min(w, x2))
        y1, y2 = max(0, min(h, y1)), max(0, min(h, y2))
        if x2 <= x1 or y2 <= y1:
            dropped += 1
            continue
        lines.append(f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
    return lines, dropped


def export_fold(fold, images_root, out, mode="copy", meta_root=ROOT, limits=None, seed=0):
    """limits: {role: n} keeps n images per role, half positive / half negative where possible (smoke tests)."""
    meta_root, images_root, out = Path(meta_root), Path(images_root), Path(out)
    mf = meta_root / "data" / "manifest.csv"
    man = pd.read_csv(mf if mf.is_file() else meta_root / "manifest.csv").set_index("path")  # repo or Kaggle layout
    sp = pd.read_csv(meta_root / "splits" / f"fold_{fold}.csv")
    assert set(sp.role) <= set(ROLES) and sp.path.is_unique
    rows, n_dropped = [], 0
    for role in ROLES:
        d = sp[sp.role == role]
        if limits and role in limits and limits[role] < len(d):
            pos = d[man.loc[d.path, "n_gt"].values > 0]
            neg = d.drop(pos.index)
            k = limits[role]
            npos = min(len(pos), k // 2 + max(0, k // 2 - len(neg)))
            d = pd.concat([pos.sample(npos, random_state=seed), neg.sample(k - npos, random_state=seed)])
        (out / "images" / role).mkdir(parents=True, exist_ok=True)
        (out / "labels" / role).mkdir(parents=True, exist_ok=True)
        for r in d.itertuples():
            m = man.loc[r.path]
            name = r.path.replace("/", "__")
            src = (images_root / r.path).resolve()
            if not src.is_file():
                sys.exit(f"missing image: {src}")
            dst = out / "images" / role / name
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            if mode == "link":
                os.symlink(src, dst)
            else:
                shutil.copyfile(src, dst)
            lines, dr = yolo_lines(json.loads(m.gt_boxes), int(m.width), int(m.height))
            n_dropped += dr
            (out / "labels" / role / (Path(name).stem + ".txt")).write_text("\n".join(lines) + ("\n" if lines else ""))
            rows.append(dict(name=name, orig_path=r.path, center=r.center, cluster_id=r.cluster_id, role=role,
                             width=int(m.width), height=int(m.height), n_gt=int(m.n_gt)))
    pd.DataFrame(rows).to_csv(out / "index.csv", index=False)
    yaml_txt = f"path: {out.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: polyp\n"
    (out / "data.yaml").write_text(yaml_txt)
    print(f"exported {fold}: {len(rows)} images -> {out} ({mode}); boxes dropped after clipping: {n_dropped}")
    return n_dropped


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", required=True)
    ap.add_argument("--images-root", required=True, help="dir containing data_C1/.. (PolypGen2021_MultiCenterData_v3)")
    ap.add_argument("--out", required=True)
    g = ap.add_mutually_exclusive_group()
    g.add_argument("--link", action="store_true", help="symlink images (Linux/Kaggle)")
    g.add_argument("--copy", action="store_true", help="copy images (default; Windows)")
    ap.add_argument("--meta-root", default=str(ROOT), help="dir with splits/ and manifest.csv (or data/manifest.csv)")
    a = ap.parse_args()
    export_fold(a.fold, a.images_root, a.out, "link" if a.link else "copy", a.meta_root)


In [ ]:
%%writefile /kaggle/working/train_fold.py
"""Train YOLO11 on one LOCO fold, then export ALL predictions (conf 0.001) for val/cal/target.

Kaggle cell:  !python train_fold.py --fold C4 --seed 0 --images-root /kaggle/input/<ds>/PolypGen2021_MultiCenterData_v3 --meta-root /kaggle/input/<ds>
Needs export_yolo.py next to this file or in ../scripts (it is imported).
Outputs in --workdir: preds_{fold}_seed{seed}.parquet (csv.gz if pyarrow missing), run_meta.json,
best_{fold}_seed{seed}.pt, index_{fold}.csv.
"""
import argparse, gc, json, platform, shutil, sys, time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path[:0] = [str(HERE), str(HERE.parent / "scripts")]
from export_yolo import export_fold  # noqa: E402

import pandas as pd  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--fold", required=True)
ap.add_argument("--images-root", required=True)
ap.add_argument("--meta-root", default=str(HERE.parent), help="dir with splits/ and manifest.csv (or data/manifest.csv)")
ap.add_argument("--workdir", default="/kaggle/working" if Path("/kaggle/working").is_dir() else "work")
ap.add_argument("--size", default="s", choices=list("nsmlx"))
ap.add_argument("--imgsz", type=int, default=640)
ap.add_argument("--epochs", type=int, default=50)
ap.add_argument("--patience", type=int, default=10)
ap.add_argument("--batch", type=int, default=16)
ap.add_argument("--pred-batch", type=int, default=4, help="batch for val/predict (small: the trainer keeps GPU memory)")
ap.add_argument("--skip-train", action="store_true", help="reuse runs/<tag>/weights/best.pt + results.csv from --workdir, no retraining")
ap.add_argument("--workers", type=int, default=2)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--copy", action="store_true", help="copy images instead of symlinking (default: link on Linux)")
ap.add_argument("--smoke", action="store_true", help="1 epoch, 64 train / 16 other images, pipeline test")
a = ap.parse_args()

import torch  # noqa: E402
import ultralytics  # noqa: E402
from ultralytics import YOLO  # noqa: E402

work = Path(a.workdir).resolve()
work.mkdir(parents=True, exist_ok=True)
ds = work / f"yolo_{a.fold}"
tag = f"{a.fold}_seed{a.seed}"
cuda = torch.cuda.is_available()
device = 0 if cuda else "cpu"
if a.smoke:
    a.epochs, a.patience, a.batch, a.workers = 1, 1, 8, 0
    limits = dict(train=64, val=16, cal=16, target=16)
else:
    limits = None

t0 = time.time()
mode = "copy" if (a.copy or platform.system() == "Windows") else "link"
export_fold(a.fold, a.images_root, ds, mode, a.meta_root, limits, a.seed)
t_export = time.time() - t0
idx = pd.read_csv(ds / "index.csv")
shutil.copy(ds / "index.csv", work / f"index_{a.fold}.csv")

best = work / "runs" / tag / "weights" / "best.pt"
if a.skip_train:
    assert best.exists(), f"--skip-train needs {best}"
    t_train = float(pd.read_csv(work / "runs" / tag / "results.csv").rename(columns=str.strip)["time"].iloc[-1])
else:
    model = YOLO(f"yolo11{a.size}.pt")
    t0 = time.time()
    model.train(data=str(ds / "data.yaml"), epochs=a.epochs, patience=a.patience, imgsz=a.imgsz, batch=a.batch,
                workers=a.workers, seed=a.seed, amp=True, device=device, project=str(work / "runs"), name=tag,
                exist_ok=True, plots=False, verbose=False)
    t_train = time.time() - t0
    del model
gc.collect()
if cuda:
    torch.cuda.empty_cache()  # the trainer leaves ~12 GB reserved on a 15 GB GPU; free it before predicting
shutil.copy(best, work / f"best_{tag}.pt")

# best epoch = argmax fitness (0.1*mAP50 + 0.9*mAP50-95) in results.csv, as Ultralytics picks best.pt
res = pd.read_csv(work / "runs" / tag / "results.csv")
res.columns = [c.strip() for c in res.columns]
fit = 0.1 * res["metrics/mAP50(B)"] + 0.9 * res["metrics/mAP50-95(B)"]
ep_times = [round(x, 2) for x in res["time"].diff().fillna(res["time"]).tolist()]  # cumulative train+val seconds -> per epoch
for i, x in enumerate(ep_times, 1):
    print(f"[time] epoch {i} {x:.1f}s", flush=True)
best_epoch = int(res["epoch"].iloc[fit.values.argmax()])

model = YOLO(str(best))
v = model.val(data=str(ds / "data.yaml"), split="val", imgsz=a.imgsz, batch=a.pred_batch, workers=a.workers,
              device=device, plots=False, verbose=False)
val_map = dict(mAP50=float(v.box.map50), mAP50_95=float(v.box.map))

rows, t_pred = [], {}
for role in ["val", "cal", "target"]:
    sub = idx[idx.role == role].set_index("name")
    t0 = time.time()
    paths = [str(ds / "images" / role / n) for n in sub.index]
    for r in model.predict(paths, conf=0.001, iou=0.6, max_det=100, imgsz=a.imgsz, batch=a.pred_batch,
                           device=device, half=cuda, stream=True, verbose=False):
        name = Path(r.path).name
        # r.boxes.xyxy is in ORIGINAL image pixels (Ultralytics rescales from the letterboxed input)
        assert tuple(r.orig_shape) == (sub.loc[name, "height"], sub.loc[name, "width"]), name
        for (x1, y1, x2, y2), s in zip(r.boxes.xyxy.cpu().tolist(), r.boxes.conf.cpu().tolist()):
            rows.append((sub.loc[name, "orig_path"], role, sub.loc[name, "center"], x1, y1, x2, y2, s))
    t_pred[role] = round(time.time() - t0, 2)
    print(f"[time] predict {role}: {len(sub)} images {t_pred[role]:.1f}s", flush=True)
df = pd.DataFrame(rows, columns=["orig_path", "role", "center", "x1", "y1", "x2", "y2", "score"])
try:
    pred_file = work / f"preds_{tag}.parquet"
    df.to_parquet(pred_file, index=False)
except ImportError:
    pred_file = work / f"preds_{tag}.csv.gz"
    df.to_csv(pred_file, index=False)

meta = dict(fold=a.fold, args=vars(a), versions=dict(python=platform.python_version(), torch=torch.__version__,
            ultralytics=ultralytics.__version__, pandas=pd.__version__),
            gpu=torch.cuda.get_device_name(0) if cuda else None, device=str(device),
            n_images=idx.role.value_counts().to_dict(), n_pred_boxes=df.role.value_counts().to_dict(),
            seconds=dict(export=round(t_export, 1), train_total=round(t_train, 1), epochs=ep_times,
                         mean_epoch=round(sum(ep_times) / max(1, len(ep_times)), 2), predict=t_pred),
            epochs_run=len(res), best_epoch=best_epoch, val=val_map, pred_file=pred_file.name)
for n in ("run_meta.json", f"run_meta_{tag}.json"):  # tagged copy survives running several seeds in one session
    (work / n).write_text(json.dumps(meta, indent=1))
print(json.dumps(meta, indent=1))


In [ ]:
# ===== CONFIG (generated; nothing to edit) =====
FOLDS = ["C1", "C2", "C3", "C4", "C5", "C6"]
SEEDS = [0]
ALREADY_DONE = set()          # runs to skip (empty: all 6 folds are trained)
EPOCHS, PATIENCE, SIZE = 50, 10, "s"   # fixed for all folds/seeds
STOP_NEW_RUNS_AFTER_H = 8.0          # never start a new run after this many hours (session limit safety)
# ===============================================================
import json, shutil, time
import pandas as pd

W = Path("/kaggle/working"); RES = W / "results"; RES.mkdir(exist_ok=True)
runs = [(f, s) for s in SEEDS for f in FOLDS if (f, s) not in ALREADY_DONE]
t_session = time.time(); fails = 0
for fold, seed in runs:
    tag = f"{fold}_seed{seed}"; d = RES / tag
    if (d / f"run_meta_{tag}.json").exists():
        print("skip (done):", tag); continue
    if (time.time() - t_session) / 3600 > STOP_NEW_RUNS_AFTER_H:
        print("time guard: not starting", tag); break
    d.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, str(W / "train_fold.py"), "--fold", fold, "--seed", str(seed), "--size", SIZE,
           "--epochs", str(EPOCHS), "--patience", str(PATIENCE),
           "--images-root", f"{DS}/PolypGen2021_MultiCenterData_v3", "--meta-root", DS]
    t0 = time.time(); print(f"=== {tag} start", time.strftime("%H:%M:%S"), flush=True)
    env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    with open(d / "log.txt", "w") as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            lf.write(line)
            if "[time]" in line or "Traceback" in line or "Error" in line:
                print(line.rstrip()[:300], flush=True)
        rc = p.wait()
    ok = rc == 0 and (W / f"run_meta_{tag}.json").exists()
    if ok:
        for pat in (f"preds_{tag}.*", f"run_meta_{tag}.json", f"best_{tag}.pt"):
            for f in W.glob(pat):
                shutil.move(str(f), d / f.name)
        shutil.copy(W / f"index_{fold}.csv", d / f"index_{fold}.csv")
        shutil.copy(W / "runs" / tag / "results.csv", d / "results.csv")
        fails = 0
    else:
        fails += 1
        print(f"!!! {tag} FAILED (rc={rc}); log: {d/'log.txt'}", flush=True)
    shutil.rmtree(W / "runs" / tag, ignore_errors=True)      # keep /kaggle/working small
    shutil.rmtree(W / f"yolo_{fold}", ignore_errors=True)
    (W / "run_meta.json").unlink(missing_ok=True)
    print(f"=== {tag} {'OK' if ok else 'FAILED'} in {(time.time()-t0)/60:.1f} min "
          f"(session {(time.time()-t_session)/3600:.2f} h)", flush=True)
    if fails >= 2:
        print("two failures in a row: stopping to protect the GPU quota"); break

rows = []
for m in sorted(RES.glob("*/run_meta_*.json")):
    j = json.loads(m.read_text())
    rows.append(dict(fold=j["fold"], seed=j["args"]["seed"], epochs_run=j["epochs_run"], best_epoch=j["best_epoch"],
                     train_min=round(j["seconds"]["train_total"] / 60, 1), val_mAP50=round(j["val"]["mAP50"], 4),
                     val_mAP50_95=round(j["val"]["mAP50_95"], 4), gpu=j["gpu"], boxes=sum(j["n_pred_boxes"].values())))
summ = pd.DataFrame(rows); summ.to_csv(RES / "summary.csv", index=False); print(summ.to_string(index=False))
name = "results_seeds_" + "_".join(map(str, SEEDS))
shutil.make_archive(str(W / name), "zip", RES)
print("download:", W / f"{name}.zip")
